# 1 Concatenate S2S

---

Concatenates all files into one large `zarr` file for anomaly generation.

Initializations run from 1999-2022. 11 ensemble members. Weekly inits.

Start the `dask` cluster first by running `cluster.ipynb`. The cell below connects to it.

You'll then open the dask tab on the left and type `proxy/####` where `####` is the port shown by `client.dashboard_link`. I then usually drag "Dask Graph" and "Dask Progress" to the right side of the notebook. Also "Dask Workers" is nice.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import cftime
import numpy as np
import xarray as xr
import os
import glob
import shutil
from tqdm import tqdm

In [2]:
from dask_cluster import get_client
client = get_client()

Grab list of file names.

In [3]:
var_map  = {"pr_sfc":    {"p": "p1", "short": "tp",     "idx": 4},
            "rlut":      {"p": "p1", "short": "rlut",   "idx": 3},
            "tas_2m":    {"p": "p1", "short": "t2m",    "idx": 4},
            "ts":        {"p": "p1", "short": "ts",     "idx": 3},
            "ua_200":    {"p": "p1", "short": "u200",   "idx": 4},
            "ua_850":    {"p": "p1", "short": "u850",   "idx": 4},
            "va_200":    {"p": "p1", "short": "v200",   "idx": 4},
            "va_850":    {"p": "p1", "short": "v850",   "idx": 4},
            "zg_200":    {"p": "p1", "short": "gh_200", "idx": 4},
            "zg_500":    {"p": "p1", "short": "gh_500", "idx": 4},
            "psl":       {"p": "p2", "short": "psl",    "idx": 3},
            "cape":      {"p": "p2", "short": "cape",   "idx": 3},
            "hfss_sfc":  {"p": "p2", "short": "hfss",   "idx": 4},
            "mrro":      {"p": "p2", "short": "mrro",   "idx": 3},
            "rzsm":      {"p": "p2", "short": "rzsm",   "idx": 3},
            "snc":       {"p": "p2", "short": "snc",    "idx": 3},
            "sty_sfc":   {"p": "p2", "short": "sty",    "idx": 4},
            "tasmin_2m": {"p": "p2", "short": "tmin",   "idx": 4},
            "uvas":      {"p": "p2", "short": "uvas",   "idx": 3},
            "wap_500":   {"p": "p2", "short": "w500",   "idx": 4},
            "hfls_sfc":  {"p": "p2", "short": "hfls",   "idx": 4},
            "huss_850":  {"p": "p2", "short": "q850",   "idx": 4},
            "mrso":      {"p": "p2", "short": "mrso",   "idx": 3},
            "rad_sfc":   {"p": "p2", "short": "rad",    "idx": 4},
            "sic":       {"p": "p2", "short": "sic",    "idx": 3},
            "stx_sfc":   {"p": "p2", "short": "stx",    "idx": 4},
            "tasmax_2m": {"p": "p2", "short": "tmax",   "idx": 4},
            "ua_100":    {"p": "p2", "short": "u100",   "idx": 4},
            "va_100":    {"p": "p2", "short": "v100",   "idx": 4}}

def update_leads(x):
    """Converts from time coordinates to lead coordinates (case-insensitive)."""
    time_dim = [d for d in x.dims if d.lower() == "time"]
    if time_dim:
        x = x.rename({time_dim[0]: "lead"})
    x = x.assign_coords(lead=np.arange(x.lead.size).astype(int))
    return x

In [4]:
model = "mpass2s_60km" # cesm2_era5, cam6_era5, mpass2s_60km, mpass2s_120km, mpass2s_60_15km, mpass2s_60_4km
modelout = "MPAS_60km"
grid = "latlon"
var_list = ["tas_2m","pr_sfc","zg_500","psl"]  # variables to process, for reference
var = var_list[0]  # pick one from var_list, then run all the cells below; repeat for the next variable

In [5]:
print(f"Running {var}")
var_info = var_map[var]
p_ver = var_info["p"]
var2 = var_info["short"]

if "mpas" in model:
    source_base = f"/lustre/desc1/espat/s2s/mpas/{model}/{model}_{grid}/{p_ver}/{var}"
else:
    source_base = f"/lustre/desc1/espat/s2s/cesm/{model}/{model}/{p_ver}/{var}"
filepath = os.path.expandvars(f"$SCRATCH/tmp/{var}/")
data_out = "output/"

Running tas_2m


In [6]:
if os.path.exists(filepath):
    shutil.rmtree(filepath)
os.makedirs(filepath)
!ln -sf {source_base}/????/??/{var}* {filepath}
files = sorted(glob.glob(os.path.join(filepath, "*.nc")))
initializations = np.unique([os.path.basename(f).split("_00z")[0].split("_")[-1] for f in files])

print(f"Finished. Unique initializations for {var}: {len(initializations)} init[0] = {initializations[0]}")

Finished. Unique initializations for tas_2m: 1252 init[0] = 01apr2002


The below loop goes through each initialization and loads in all 10 ensemble members. It converts dimension names to the appropriate names for `climpred` and then chunks them into 100MB chunks in `dask`. We append it all to one big python list which will be used to concatenate into a large dataset with `(init, lead, member, lat, lon)`. This took about 2 minutes for me to run.

In [7]:
S2Sensemble = []
for init in tqdm(initializations):
    if grid=="native":
        filelist = np.sort(glob.glob(filepath + f"{var}*{init}_00z_unst_d*.nc"))
        chunks = {"lead": -1, "ncells": -1}
    elif grid=="latlon":
        filelist = np.sort(glob.glob(filepath + f"{var}*{init}_00z_d01_d*.nc"))
        chunks = {"lead": -1, "lat": -1, "lon": -1}
    ds = xr.open_mfdataset(filelist,combine="nested",parallel=True,concat_dim="member",preprocess=update_leads,
                           chunks = chunks,coords="minimal",compat="override",)

    MON_TO_INT = {"jan": 1,"feb": 2,"mar": 3,"apr": 4,"may": 5,"jun": 6,"jul": 7,"aug": 8,"sep": 9,"oct": 10,"nov": 11,"dec": 12}

    day = init[0:2]
    mon = init[2:5]; mon = MON_TO_INT[mon]
    year = init[5::]

    ds = ds.assign_coords(lead=np.arange(ds.lead.size).astype(int),
                          init=cftime.DatetimeProlepticGregorian(int(year),int(mon),int(day)),
                          member=np.arange(ds[var].shape[0]) + 1,)

    ds = ds.chunk({"member": -1})
    S2Sensemble.append(ds)

100%|██████████| 1252/1252 [10:14<00:00,  2.04it/s]


Now we can concatenate into one big dataset. Keep in mind we've used `dask` this whole time so everything is out of memory. (Should take seconds or less)

In [8]:
ds = xr.concat(S2Sensemble,dim="init",coords="minimal",compat="override",combine_attrs="override")

In [9]:
ds = ds.sortby("init")
ds["init"] = ds.indexes["init"].to_datetimeindex(time_unit="ns")

In [10]:
ds = ds.rename({var: var2})
if var=="pr_sfc":
    ds[var2] = ds[var2]*86400; ds[var2] = ds[var2].astype("float32")
display(ds[var2])

<xarray.DataArray 't2m' (init: 1252, member: 11, lead: 46, lat: 181, lon: 360)> Size: 165GB
dask.array<getitem, shape=(1252, 11, 46, 181, 360), dtype=float32, chunksize=(1, 11, 46, 181, 360), chunktype=numpy.ndarray>
Coordinates:
  * lon      (lon) float64 3kB 0.0 1.0 2.0 3.0 4.0 ... 356.0 357.0 358.0 359.0
  * lat      (lat) float64 1kB -90.0 -89.0 -88.0 -87.0 ... 87.0 88.0 89.0 90.0
  * lead     (lead) int64 368B 0 1 2 3 4 5 6 7 8 ... 37 38 39 40 41 42 43 44 45
  * init     (init) datetime64[ns] 10kB 1999-01-04 1999-01-11 ... 2022-12-26
  * member   (member) int64 88B 1 2 3 4 5 6 7 8 9 10 11
Attributes:
    long_name:  2-meter temperature
    units:      K

In [42]:
%time ds.to_zarr(f"{data_out}{modelout}.raw.daily.geospatial.zarr",mode='a',consolidated=True)

CPU times: user 1.61 s, sys: 50.7 ms, total: 1.67 s
Wall time: 58.7 s
